### Tools

In [13]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-2.5-flash")

response=model.invoke("What is FIFA ?")
response

AIMessage(content="**FIFA** stands for **Fédération Internationale de Football Association**.\n\nIt is the **international governing body of association football** (also known as soccer), futsal, and beach soccer.\n\nHere's a breakdown of what FIFA is and what it does:\n\n1.  **Governing Body:** It is the highest authority in world football, responsible for overseeing and regulating the sport globally.\n2.  **Headquarters:** It is headquartered in Zurich, Switzerland.\n3.  **Foundation:** It was founded in Paris on May 21, 1904.\n4.  **Membership:** FIFA comprises 211 national football associations across the globe, each representing football in its respective country. This gives it more member countries than the United Nations!\n5.  **Key Functions:**\n    *   **Organizing Tournaments:** Its most famous responsibility is organizing the **FIFA World Cup** (for men) and the **FIFA Women's World Cup**, which are the most prestigious international football tournaments in the world. It als

In [14]:
### Tools

from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location."""
    return f"It's sunny in {location}"


model_with_tool=model.bind_tools([get_weather])

In [15]:
response=model_with_tool.invoke("Whats the weather in Bangalore?")
print(response)

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Bangalore"}'}, '__gemini_function_call_thought_signatures__': {'017f7df2-e9d1-423d-9ac4-f453d76eb586': 'Cu8BAQw51seuMO2NidP2PyqeCWIrwGqWJGWkLwCPG3VBi1f7GK7mOcbYbT/cul9RMx+V5U/EOCZuSa6crS1+xFkLL17W9s7YeqgAVAOcVP9hPFj5k08rakVLJ6ai70TQeX4I4WhVOg92VYNg3khd6br/BxyUUtT2kO83qs6lTLCbOubsqG63zFpgd0tMIg34nY/llnJNiv5/pxn41TDq6nCq0Wr9dHyUXTPoihzHZ/h73h/x7cM7Woi0q11Qo/hFl8uCAyZQ5ldq2SnMgfVlhK5tjTMLLtCMXBMP0sYPgml9w4Ogr0HCy+MQnAIsVbWFjFo='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f1d3a-9575-7173-88a7-fa43d3a2e56f-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': '017f7df2-e9d1-423d-9ac4-f453d76eb586', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 46, 'output_tokens': 63, 'total_tokens': 109, 'input_token_details': {'cache_read': 0}, 'o

In [16]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Bangalore'},
  'id': '017f7df2-e9d1-423d-9ac4-f453d76eb586',
  'type': 'tool_call'}]

### Tool Execution Loop

In [17]:
# Step 1: Model generates tool calls
messages=[{"role":"user","content":"What's the weather in Bangalore?"}]
ai_msg=model_with_tool.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results (i.e context)
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response

final_response=model_with_tool.invoke(messages)
print(final_response.text)

The weather in Bangalore is sunny.


In [18]:
from langchain_core.messages import HumanMessage

# Step 1: Model generates tool calls
messages = [HumanMessage("What's the weather in Bangalore?")]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# Check the model actually asked for the tool
print("Tool calls:", ai_msg.tool_calls)

# Step 2: Execute each requested tool, append the result
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)   # returns a ToolMessage
    messages.append(tool_result)

# Step 3: Model produces final answer using the tool result
final_response = model_with_tool.invoke(messages)
print(final_response.content)

Tool calls: [{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': 'ba32218c-6174-4345-85d2-e8258e916112', 'type': 'tool_call'}]
It's sunny in Bangalore.


In [19]:
messages

[HumanMessage(content="What's the weather in Bangalore?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Bangalore"}'}, '__gemini_function_call_thought_signatures__': {'ba32218c-6174-4345-85d2-e8258e916112': 'Cu8BAQw51sf0yzBVL0jZi+SVIySpW4MNlqW/Sm5ScwE6rf33K7dWOyLESidg/7lQjSmzAbzRkRUooLRKMmgM5qYMYlZ/3Yh4HhEcH43sy0pjll8KfRCisVj+L9+5xIbmMeLDhibt8DvkCvqS96PU3joTCYVcPgT0KT4Mxc45CNntE/0ESeFW0YIHigHMGKQCyuh1385ceaUdbUBGuPE4pLwM9H6DJVQfECaCjPWDsowjYqKenefnwn1tQonOnqED0Iwm3uka9hJzmCV/xd22mOZz4S8/Eekw6OMYl1FtO1fDFEjyijZD6JW5DBdq3oCBCHw='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1d3a-d728-7890-a0cc-9af65aa59aea-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': 'ba32218c-6174-4345-85d2-e8258e916112', 'type': 'tool_call'}], invalid_tool_calls=[], us

In [20]:
model_with_tool

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-google-genai': '4.2.6'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x11a7378c0>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weath